# Chương 15: Tính minh bạch và Khả năng giải thích của AI (XAI)

Notebook này minh họa thực nghiệm các thuật toán giải thích mô hình cốt lõi nhất từ các phép toán cơ bản bằng NumPy:
1. **Cài đặt LIME (Local Interpretable Model-agnostic Explanations) từ đầu**: Lấy mẫu nhiễu cục bộ, gán trọng số khoảng cách qua hàm nhân Exponential Kernel, huấn luyện mô hình tuyến tính đại diện có trọng số (Weighted Linear Surrogate).
2. **Tính toán chính xác giá trị Shapley (SHAP) theo lý thuyết trò chơi**: Duyệt toàn bộ $2^M$ liên minh để phân bổ tiền thưởng công bằng, kiểm chứng 4 tiên đề Shapley.
3. **So sánh Bản đồ nổi bật (Saliency Map) vs Đạo hàm tích phân (Integrated Gradients)**: Kiểm chứng bằng thực nghiệm tiên đề Tính đầy đủ (Completeness Axiom) và giải quyết hiện tượng bão hòa gradient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import math

# Khởi tạo seed
np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Cài đặt LIME từ con số không (LIME from scratch)

### Nguyên lý thuật toán
Cho một mô hình hộp đen phức tạp $f(x)$ và điểm dữ liệu cần giải thích $x^* \in \mathbb{R}^d$:
1. Sinh $N$ điểm nhiễu lân cận: $z_k \sim \mathcal{N}(x^*, \sigma^2 I)$.
2. Tính trọng số khoảng cách bằng nhân hàm mũ: $\pi(z_k) = \exp\left(-\frac{\|x^* - z_k\|_2^2}{2 \sigma^2}\right)$.
3. Thu thập dự đoán từ hộp đen: $y_k = f(z_k)$.
4. Huấn luyện mô hình hồi quy tuyến tính có trọng số $\min_w \sum_{k=1}^N \pi(z_k) (w^T z_k - y_k)^2 + \lambda \|w\|_2^2$.

In [ ]:
# Định nghĩa một hàm hộp đen phi tuyến phức tạp mô phỏng duyệt tín dụng
# x0: Thu nhập (triệu VNĐ), x1: Điểm tín dụng, x2: Tỷ lệ nợ/thu nhập
def black_box_model(X):
    x0, x1, x2 = X[:, 0], X[:, 1], X[:, 2]
    # Tương tác phi tuyến mạnh
    logit = 0.5 * x0 + 1.2 * x1 - 2.5 * x2 + 0.3 * (x0 * x1) - 0.5 * (x2**2)
    prob = 1.0 / (1.0 + np.exp(-logit))
    return prob

def explain_instance_lime(model_fn, x_star, num_samples=1000, kernel_width=1.0, alpha_reg=1e-3):
    dim = len(x_star)
    # 1. Sinh các điểm nhiễu xung quanh x_star
    perturbations = np.random.randn(num_samples, dim) * kernel_width
    samples = x_star + perturbations
    
    # 2. Tính khoảng cách Euclid và trọng số hạt nhân (Kernel weights)
    distances = np.linalg.norm(perturbations, axis=1)
    weights = np.exp(-(distances ** 2) / (kernel_width ** 2))
    
    # 3. Lấy nhãn dự đoán từ hộp đen
    y_labels = model_fn(samples)
    
    # 4. Hồi quy Ridge có trọng số (Weighted Ridge Regression)
    # Thêm cột bias 1 vào ma trận đầu vào
    Z = np.hstack([np.ones((num_samples, 1)), perturbations])  # giải thích theo độ lệch so với x_star
    W_diag = np.diag(weights)
    
    # Nghiệm đóng: w = (Z^T W Z + lambda I)^(-1) Z^T W y
    A = Z.T @ (weights[:, None] * Z) + alpha_reg * np.eye(dim + 1)
    b = Z.T @ (weights * y_labels)
    coefs = np.linalg.solve(A, b)
    
    intercept = coefs[0]
    feature_importance = coefs[1:]
    return intercept, feature_importance

# Kiểm thử với một khách hàng cụ thể
x_customer = np.array([2.0, 1.5, 0.8])  # [Thu nhập cao, Điểm tín dụng tốt, Nợ trung bình]
pred_prob = black_box_model(x_customer.reshape(1, -1))[0]
intercept, attributions = explain_instance_lime(black_box_model, x_customer)

print(f"Xác suất được duyệt bởi Black-Box: {pred_prob:.4f}")
print(f"Hệ số chặn cục bộ (Local Intercept): {intercept:.4f}")
features = ["Thu nhập (x0)", "Điểm tín dụng (x1)", "Tỷ lệ nợ (x2)"]
for feat, val in zip(features, attributions):
    print(f" - Tầm quan trọng LIME của [{feat}]: {val:+.4f}")

## 2. Tính toán chính xác giá trị Shapley (SHAP) theo Lý thuyết trò chơi

### Công thức giá trị Shapley
$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|! \, (|F| - |S| - 1)!}{|F|!} \left[ f(S \cup \{i\}) - f(S) \right]$$
Kiểm chứng 4 tiên đề cốt lõi:
1. **Tính hiệu quả (Efficiency)**: $\sum_{i=1}^M \phi_i = f(x^*) - \mathbb{E}[f]$.
2. **Tính đối xứng (Symmetry)**.
3. **Người chơi bù nhìn (Dummy Player)**: Thuộc tính không đóng góp gì thì $\phi_i = 0$.
4. **Tính cộng tính (Additivity)**.

In [ ]:
# Tạo một hàm giá trị trò chơi trên 3 người chơi F = {0, 1, 2}
# Giá trị kỳ vọng nền E[f] khi không có người chơi nào tham gia là f(empty_set) = 10
baseline_val = 10.0

# Bảng giá trị hợp tác cho từng liên minh con (đại diện cho hàm f(S))
coalition_values = {
    (): baseline_val,
    (0,): 25.0,
    (1,): 15.0,
    (2,): baseline_val,         # Người chơi 2 là Dummy (không đóng góp gì)
    (0, 1): 50.0,              # Tương tác tương hỗ mạnh giữa 0 và 1
    (0, 2): 25.0,
    (1, 2): 15.0,
    (0, 1, 2): 50.0            # Người chơi 2 vẫn không làm tăng giá trị
}

def compute_exact_shapley(values_dict, num_players=3):
    all_players = set(range(num_players))
    shapley_values = np.zeros(num_players)
    
    for i in range(num_players):
        other_players = list(all_players - {i})
        phi_i = 0.0
        
        # Duyệt qua tất cả các kích thước tập con từ 0 đến M-1
        for k in range(len(other_players) + 1):
            weight = (math.factorial(k) * math.factorial(num_players - k - 1)) / math.factorial(num_players)
            
            for subset in combinations(other_players, k):
                S = tuple(sorted(subset))
                S_with_i = tuple(sorted(subset + (i,)))
                
                marginal_contribution = values_dict[S_with_i] - values_dict[S]
                phi_i += weight * marginal_contribution
                
        shapley_values[i] = phi_i
        
    return shapley_values

shapley_vals = compute_exact_shapley(coalition_values, num_players=3)

print("Giá trị Shapley tính toán chính xác:")
for p_idx, val in enumerate(shapley_vals):
    print(f" - Người chơi {p_idx}: {val:.2f}")

# Kiểm chứng Tiên đề 1: Tính hiệu quả (Efficiency Axiom)
payout_grand_coalition = coalition_values[(0, 1, 2)] - coalition_values[()]
sum_shapley = np.sum(shapley_vals)
print(f"\nKiểm chứng Tiên đề Hiệu quả: Tổng Shapley = {sum_shapley:.2f} == Payout thực tế = {payout_grand_coalition:.2f} -> {np.isclose(sum_shapley, payout_grand_coalition)}")
print(f"Kiểm chứng Người chơi bù nhìn (Dummy Player 2): phi_2 = {shapley_vals[2]:.2f} == 0 -> {np.isclose(shapley_vals[2], 0.0)}")

## 3. Bản đồ nổi bật (Saliency) vs Đạo hàm tích phân (Integrated Gradients)

Kiểm chứng hiện tượng bão hòa gradient (gradient saturation) và chứng minh **Integrated Gradients** thỏa mãn tiên đề Tính đầy đủ (Completeness):
$$\sum_{i=1}^d \text{IG}_i(x) = f(x) - f(x')$$ 
trong khi phương pháp đạo hàm trực tiếp thô (Vanilla Saliency $\nabla_x f(x) \odot (x - x')$) bị sai lệch nghiêm trọng.

In [ ]:
# Mô hình phi tuyến có hiện tượng bão hòa với hàm Softplus / ReLU
def saturated_model(x):
    # f(x) = softplus(2 * x0 + 0.1 * x1 - 1.0)
    z = 2.0 * x[0] + 0.1 * x[1] - 1.0
    return np.log(1.0 + np.exp(z))

def model_grad(x):
    z = 2.0 * x[0] + 0.1 * x[1] - 1.0
    sig = 1.0 / (1.0 + np.exp(-z))
    grad_x0 = 2.0 * sig
    grad_x1 = 0.1 * sig
    return np.array([grad_x0, grad_x1])

# Điểm cơ sở (baseline) x' và điểm quan sát x
x_baseline = np.array([0.0, 0.0])
x_target = np.array([3.0, 1.0])

f_baseline = saturated_model(x_baseline)
f_target = saturated_model(x_target)
actual_diff = f_target - f_baseline

# 1. Vanilla Saliency
raw_grad = model_grad(x_target)
saliency_attr = raw_grad * (x_target - x_baseline)

# 2. Integrated Gradients qua xấp xỉ tổng Riemann 100 bước
m_steps = 100
alphas = np.linspace(0, 1, m_steps)
accum_grads = np.zeros(2)

for alpha in alphas:
    x_step = x_baseline + alpha * (x_target - x_baseline)
    accum_grads += model_grad(x_step)

ig_attr = (x_target - x_baseline) * (accum_grads / m_steps)

print(f"Chênh lệch thực tế Delta f = f(x) - f(x') = {actual_diff:.6f}")
print(f"Tổng phân bổ của Vanilla Saliency: {np.sum(saliency_attr):.6f} (Sai lệch: {abs(np.sum(saliency_attr) - actual_diff):.6f})")
print(f"Tổng phân bổ của Integrated Gradients: {np.sum(ig_attr):.6f} (Sai lệch: {abs(np.sum(ig_attr) - actual_diff):.6f})")

# Trực quan hóa so sánh sự đóng góp của các thuộc tính
fig, ax = plt.subplots(figsize=(6, 4))
bar_width = 0.35
idx = np.arange(2)

ax.bar(idx - bar_width/2, saliency_attr, bar_width, label="Vanilla Saliency", color="salmon")
ax.bar(idx + bar_width/2, ig_attr, bar_width, label="Integrated Gradients (IG)", color="teal")
ax.set_xticks(idx)
ax.set_xticklabels(["Thuộc tính x0", "Thuộc tính x1"])
ax.set_ylabel("Mức độ đóng góp (Attribution)")
ax.set_title("So sánh Saliency vs Integrated Gradients (Khắc phục bão hòa)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Hoàn tất thực hành Chapter 15: Kiểm chứng thành công LIME, SHAP và Integrated Gradients!")